# 10 · Continuous vs discretised numeric input (paper §2.2, Fig. 2)

The paper argues against discretising numeric features: each value scales its own token
embedding instead. This compares the same model on the same 5 splits of METABRIC and
SUPPORT with standardised values vs 10 quantile bins (the older `metabric`/`support`
parsers), for our model (L_PCH) and DeepHit. Output: paired differences
(`continuous_vs_discretised_*.csv`).

## Options

In [ ]:
NAME = "10_continuous_vs_discretised"

# Paper §2.2 / Fig. 2: numeric values enter through the numeric-value embedding instead
# of being discretised. Same model, same splits; only the feature encoding differs.
DATASETS = {
    #            continuous (standardised)               discretised (10 quantile bins)
    "metabric": (["dataset=metabric_numeric"],          ["data/load=metabric", "data/parse=metabric",
                                                         "tokenizers=metabric", "dataset=metabric"]),
    "support":  (["dataset=support_numeric"],           ["data/load=support", "data/parse=support",
                                                         "tokenizers=support", "dataset=support"]),
}
GROUP  = {"metabric": "survtrace_metabric", "support": "survtrace_support"}
MODELS = [("ours", "nllpch"), ("deephit_paper", None)]
SEEDS  = [0, 1, 2, 3, 4]

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/version_1/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Prepare both encodings

In [ ]:
if SMOKE_TEST:
    SEEDS = SEEDS[:1]
ENC = {}
for ds, (cont, disc) in DATASETS.items():
    for enc, ov in [("continuous", cont), ("discretised", disc)]:
        ENC[(ds, enc)] = ov
        r.prepare(f"{GROUP[ds]}/survival", ov, label=f"{ds}_{enc}")

## Run

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for (ds, enc), ov in ENC.items():
        hub = [o.split("=")[1] for o in ov if o.startswith("dataset=")][0]
        base = {"dataset": ds, "seed": s, "repr": enc}
        for model, rec in MODELS:
            if model == "ours":
                runs.append(Run(f"{ds}_{enc}__ours_{rec}__s{s}", "finetune", f"{GROUP[ds]}/survival",
                                ov + so + [f"tasks/losses={rec}"], hub, base | {"model": "ours", "recipe": rec}))
            else:
                runs.append(Run(f"{ds}_{enc}__{model}__s{s}", "finetune", f"{GROUP[ds]}/{model}",
                                ov + so, hub, base | {"model": model}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

from scripts.runner import paired_vs
res = r.collect()
for metric in ["ctd_weighted_avg", "brier_survtrace_weighted_avg"]:
    t = paired_vs(res, metric, by="repr", baseline="discretised", pair_on="seed", within=["dataset", "model"])
    t.to_csv(r.results_dir / "report" / f"continuous_vs_discretised_{metric}.csv", index=False)
    display(t)

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)